In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd

import pickle

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

from rank_bm25 import BM25Okapi

from duckduckgo_search import DDGS

print(
    "Libraries Loaded Successfully"
)

Libraries Loaded Successfully


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_2710/661869527.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [4]:
# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

eval_df = pd.read_csv(
    "../notebooks/evaluation_dataset.csv"
)

print(
    "Rows:",
    len(eval_df)
)

eval_df.head()

Rows: 1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [5]:
# ============================================================
# LOAD EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(

    model="text-embedding-3-small"

)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [7]:
# ============================================================
# LOAD FAISS
# ============================================================

vector_store = FAISS.load_local(

    "../notebooks/faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [8]:
# ============================================================
# LOAD CHUNKS
# ============================================================

with open(

    "../notebooks/all_chunks.pkl",

    "rb"

) as f:

    all_chunks = pickle.load(
        f
    )

print(
    "Chunks Loaded:",
    len(all_chunks)
)

Chunks Loaded: 160571


In [9]:
# ============================================================
# TOKENIZE CHUNKS
# ============================================================

tokenized_chunks = [

    chunk.lower().split()

    for chunk in all_chunks

]

print(
    "Tokenization Completed"
)

Tokenization Completed


In [10]:
# ============================================================
# LOAD BM25
# ============================================================

bm25 = BM25Okapi(
    tokenized_chunks
)

print(
    "BM25 Loaded Successfully"
)

BM25 Loaded Successfully


In [11]:
# ============================================================
# WEB SEARCH FUNCTION
# ============================================================

def web_search(

    query,

    max_results=5

):

    results = []

    with DDGS() as ddgs:

        search_results = list(

            ddgs.text(

                query,

                max_results=max_results

            )

        )

    for item in search_results:

        results.append(

            item["body"]

        )

    return results

In [12]:
# ============================================================
# LOAD LLAMA3
# ============================================================

llm = ChatOllama(

    model="llama3",

    temperature=0

)

print(
    "Llama3 Loaded Successfully"
)

Llama3 Loaded Successfully


In [13]:
# ============================================================
# RESET RESULTS
# ============================================================

results = []

print(
    "Results Reset Successfully"
)

Results Reset Successfully


In [14]:
# ============================================================
# FUSION LLAMA3 EVALUATION PIPELINE
# ============================================================

for idx, row in eval_df.iterrows():

    if idx >= 25:
        break

    print(
        f"Processing Row {idx+1}/25"
    )

    query = str(
        row["query"]
    )

    reference_summary = str(
        row["reference_summary"]
    )

    # ========================================================
    # DENSE RETRIEVAL
    # ========================================================

    dense_docs = vector_store.similarity_search(

        query,

        k=5

    )

    dense_texts = [

        doc.page_content

        for doc in dense_docs

    ]

    # ========================================================
    # SPARSE RETRIEVAL
    # ========================================================

    tokenized_query = query.lower().split()

    sparse_indices = bm25.get_top_n(

        tokenized_query,

        list(
            range(
                len(all_chunks)
            )
        ),

        n=5

    )

    sparse_docs = [

        all_chunks[idx]

        for idx in sparse_indices

    ]

    # ========================================================
    # WEB SEARCH
    # ========================================================

    web_docs = web_search(

        query,

        max_results=5

    )

    # ========================================================
    # FUSION
    # ========================================================

    fusion_docs = (

        dense_texts

        +

        sparse_docs

        +

        web_docs

    )

    # ========================================================
    # CONTEXT
    # ========================================================

    context = "\n\n".join(
        fusion_docs
    )

    # ========================================================
    # PROMPT
    # ========================================================

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # ========================================================
    # GENERATE RESPONSE
    # ========================================================

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({

        "pipeline":
        "Fusion Llama3",

        "model":
        "Llama3",

        "retrieval_method":
        "Fusion",

        "query":
        query,

        "reference_summary":
        reference_summary,

        "generated_answer":
        answer

    })

print(
    "\nFusion Llama3 Evaluation Completed"
)

Processing Row 1/25


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_2710/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Processing Row 2/25


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_2710/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Processing Row 3/25


...

Processing Row 24/25


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_2710/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Processing Row 25/25


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_2710/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Fusion Llama3 Evaluation Completed


In [15]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 25


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Fusion Llama3,Llama3,Fusion,solar observatory aims to provide better under...,solar observatory aims to provide better under...,"According to the provided context, the Solar D..."
1,Fusion Llama3,Llama3,Fusion,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"According to the provided context, Mary J. Bli..."
2,Fusion Llama3,Llama3,Fusion,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to the provided context, German..."
3,Fusion Llama3,Llama3,Fusion,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,"According to the provided context, presidentia..."
4,Fusion Llama3,Llama3,Fusion,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"According to the provided context, China's Oly..."


In [16]:
# ============================================================
# DATAFRAME INFO
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   pipeline           25 non-null     str  
 1   model              25 non-null     str  
 2   retrieval_method   25 non-null     str  
 3   query              25 non-null     str  
 4   reference_summary  25 non-null     str  
 5   generated_answer   25 non-null     str  
dtypes: str(6)
memory usage: 15.5 KB


In [17]:
# ============================================================
# CHECK NULL VALUES
# ============================================================

results_df.isnull().sum()

pipeline             0
model                0
retrieval_method     0
query                0
reference_summary    0
generated_answer     0
dtype: int64

In [18]:
# ============================================================
# MANUAL VALIDATION
# ============================================================

results_df[

    [

        "query",

        "reference_summary",

        "generated_answer"

    ]

]

,query,reference_summary,generated_answer
0,solar observatory aims to provide better under...,solar observatory aims to provide better under...,"According to the provided context, the Solar D..."
1,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"According to the provided context, Mary J. Bli..."
2,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to the provided context, German..."
3,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,"According to the provided context, presidentia..."
4,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"According to the provided context, China's Oly..."
5,critics and viewers see stewart as victor after,critics and viewers see stewart as victor afte...,"According to the provided context, many declar..."
6,bahr idriss abu garda faces charges in deaths,bahr idriss abu garda faces charges in deaths ...,"Bahr Idriss Abu Garda faces charges of murder,..."
7,south africa beat cohosts india in world cup,south africa beat cohosts india in world cup c...,"Yes, according to the provided context, South ..."
8,rudy ruiz its become unfashionable to have an,rudy ruiz its become unfashionable to have an ...,"Based on the provided context, it appears that..."
9,fabio cannavaro is to join the italian national,fabio cannavaro is to join the italian nationa...,"According to the provided context, Fabio Canna..."


In [19]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "fusion_llama3_eval.csv",

    index=False

)

print(
    "Fusion Llama3 Evaluation Results Saved Successfully"
)

Fusion Llama3 Evaluation Results Saved Successfully


In [20]:
# ============================================================
# VERIFY SAVED FILE
# ============================================================

saved_df = pd.read_csv(
    "fusion_llama3_eval.csv"
)

print(
    saved_df.shape
)

saved_df.head()

(25, 6)


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Fusion Llama3,Llama3,Fusion,solar observatory aims to provide better under...,solar observatory aims to provide better under...,"According to the provided context, the Solar D..."
1,Fusion Llama3,Llama3,Fusion,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"According to the provided context, Mary J. Bli..."
2,Fusion Llama3,Llama3,Fusion,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to the provided context, German..."
3,Fusion Llama3,Llama3,Fusion,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,"According to the provided context, presidentia..."
4,Fusion Llama3,Llama3,Fusion,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"According to the provided context, China's Oly..."


In [21]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "Pipeline : Fusion Llama3"
)

print(
    "Rows Evaluated :",
    len(saved_df)
)

print(
    "Output File : fusion_llama3_eval.csv"
)

print(
    "Evaluation Completed Successfully"
)

Pipeline : Fusion Llama3
Rows Evaluated : 25
Output File : fusion_llama3_eval.csv
Evaluation Completed Successfully
